In [65]:
import polars as pl
import polars.selectors as cs

In [66]:
# Las siguientes variables hacen referencia a archivos que el usuario no va a subir, porque son datos de vigencias cerradas
# Los archivos de 2024 no se modifican, porque esta fue la infomación que quedó del avance de esa vigencia
archivo_2024_hacienda = "Informes Financieros/EJECUCION INVERSION A DICIEMBRE 31 DEL 2024 ENERO 10 2025.xlsx" 
archivo_2024_regalias = "Informes Financieros/INFORME FINANCIERO REGALIAS A 31 DE DICIEMBRE DE 2024.xlsx"
# Los archivos de 2025 no se modifican, porque esta fue la infomación que quedó del avance de esa vigencia
archivo_2025_hacienda = "Informes Financieros/EJECUCION INVERSION DE ENERO A DICIEMBRE 2025.xlsx"
archivo_2025_regalias = "Informes Financieros/PAGOS REGALIAS ENERO - DICIEMBRE 2025.xlsx"
archivo_recursos_propios_2025_ads = "Informes Financieros/Aguas de Sucre/RELACION DE PAGOS ENERO A DICIEMBRE ADS.xlsx"
archivo_regalias_2025_ads = "Informes Financieros/Aguas de Sucre/PAGOS REGALIAS 2025 ADS.xlsx"
archivo_gestiones_2025 = "Informes Financieros/OTRAS FUENTES DE GESTIONES/EjecucionFinancieraGestiones_20260210.xlsx"
archivo_fondo_mixto_2025 = "Informes Financieros/FONDO MIXTO/CONTRATOS Y CONVENIOS 2025 - FONDO MIXTO.xlsx"
archivo_ejecucion_financiera_inder = "Informes Financieros/Indersucre/EjecucionIndersucre_Territorial_Regalias_202602010.xlsx"

# Las siguientes variables hacen referencia a archivos que el usuario constantemente va a estar subiendo para actualizar los datos
archivo_plan_indicativo = "Plan Indicativo 2024-2027.xlsx"
archivo_2026_hacienda = "Informes Financieros/EJECUCION INVERSION DE HACIENDA PRUEBA 2026.xlsx"
archivo_2026_regalias = "Informes Financieros/CG-cttos_04_marzo_20260304.xlsx"


plan_indicativo = pl.read_excel(
    archivo_plan_indicativo,
    table_name="tblPlanIndicativo_2",
)

orden_lineas_pdd = pl.read_excel(archivo_plan_indicativo,table_name="orden_lineas")
orden_sectores_pdd = pl.read_excel(archivo_plan_indicativo,table_name="orden_sectores")
orden_programas_pdd = pl.read_excel(archivo_plan_indicativo,table_name="orden_programas")
homologacion_secretarias = pl.read_excel(archivo_plan_indicativo,table_name="HomologacionSecretarias") # La columna varias secretarías se puede usar
                                                                                                    # como filtro y no como columna de la tabla

columnas_prog_ejec_fisica = plan_indicativo.select("Codigo Meta","Línea Estratégica","Sector PDD",
               "Numero Programa PDD","Programa PDD","Indicador de producto principal","Código del indicador principal","Meta de cuatrienio",
               "Tipo de acumulación","Responsable","Meta Física Esperada 2024",
               "Meta Física Esperada 2025","Meta Física Esperada 2026","Meta Física Esperada 2027",
              "PROYECTOS 2024","PROYECTOS 2025","PROYECTOS/GESTIONES PROGRAMADAS 2026","PROYECTOS 2026",
               "PROYECTOS 2027","EJECUCIÓN 2024","PORCENTAJE DE EJECUCIÓN 2024","CATEGORÍA DE EJECUCIÓN FÍSICA 2024",
               "EJECUCIÓN 2025","PORCENTAJE DE EJECUCIÓN 2025","CATEGORÍA DE EJECUCIÓN FÍSICA 2025","EJECUCIÓN 2026",
               "PORCENTAJE DE EJECUCIÓN 2026","CATEGORÍA DE EJECUCIÓN FÍSICA 2026","EJECUCIÓN ACUMULADA",
               "PORCENTAJE DE EJECUCIÓN ACUMULADA","CATEGORÍA DE EJECUCIÓN ACUMULADA")



def programacion_financiera( vigencia:str ) :
    suma = pl.col("programación recursos propios icld" + vigencia) + pl.col("programación recursos propios icde" + vigencia)
    + pl.col("programación sgp educación" + vigencia ) + pl.col("programación sgp salud" + vigencia)
    + pl.col("programación sgp apsb" + vigencia) + pl.col("programación cofinanciación municipio" + vigencia)
    + pl.col("programación cofinanciación nación" + vigencia) + pl.col("programación crédito" + vigencia)
    + pl.col("programación regalías" + vigencia) + pl.col("programación otras fuentes" + vigencia)
    return suma

# Reporte de Ejecución Financiera

### Lectura y Anexado de Fuentes

In [67]:
# Ejecución Financiera 2024
ejecucion_regalias_2024 = (
    pl.read_excel(
        archivo_2024_regalias,
        table_name="EjecucionRegalias",
        columns=["CODIGO META","COMPROMISOS","CLASIFICACIÓN RECURSOS"]
    ).with_columns(
        pl.col("CODIGO META").fill_null(pl.lit(""))
    ).filter(
        pl.col("CODIGO META") != "" ,pl.col("CODIGO META").str.starts_with("MT")
    ).rename({"COMPROMISOS":"RP"})
)

ejecucion_hacienda_2024 = (
    pl.read_excel(
        archivo_2024_hacienda,
        table_name="EjecucionHaciendaDiciembre",
        columns=["RP", "CODIGO META", "CLASIFICACIÓN RECURSOS"]
    ).with_columns(
        pl.col("CODIGO META","CLASIFICACIÓN RECURSOS").fill_null(pl.lit(""))
    ).filter(
        pl.col("CODIGO META") != "" ,
        pl.col("CLASIFICACIÓN RECURSOS") != "" ,
    )
)

# Ejecución Financiera 2025

ejecucion_regalias_2025 = (
    pl.read_excel(
        archivo_2025_regalias,
        table_name="Pagos_Regalias_2025"
    ).select(
        "PAGOS REGALIAS","CODIGO META","CLASIFICACIÓN RECURSOS"
    ).rename( # Hago la claridad que pago no es RP, sin embargo, le cambio el nombre a la columna para no generar error cuando haga concat con ejecucion_hacienda_2025
        {"PAGOS REGALIAS":"RP"}
    ).with_columns(
        pl.col("CODIGO META").fill_null(pl.lit(""))
    ).filter(pl.col("CODIGO META") != "")
    
)

ejecucion_hacienda_2025 = (
    pl.read_excel(
        archivo_2025_hacienda,
        table_name="EjecucionHaciendaDiciembre2025",
    ).with_columns(
        pl.col("PROYECTO ARCHIVADO","CODIGO META","CLASIFICACIÓN RECURSOS","SE VA A CARGAR EN PI").fill_null(pl.lit("")),
        pl.when(pl.col("DISTRIBUIR DE FORMA EQUITATIVA")=="SI").then(pl.col("RP")/2).otherwise(pl.col("RP"))
    ).filter(
        pl.col("PROYECTO ARCHIVADO") == "",
        pl.col("CODIGO META") != "",
        pl.col("CLASIFICACIÓN RECURSOS") != "",
        pl.col("SE VA A CARGAR EN PI") == "",
    ).select("CODIGO META", "CLASIFICACIÓN RECURSOS", "RP")
)

# Estas son las ejecuciones financieras que tenia pendiente agregar de 2025

ejecucion_2025_ads_recursos_propios = (
    pl.read_excel(
        archivo_recursos_propios_2025_ads,
        table_name="PagosAguasDeSucre"
    )
    .select("VALOR DEL PAGO", "CLASIFICACIÓN RECURSOS", "CODIGO META")
    .with_columns(pl.col("CODIGO META").fill_null(pl.lit("")))
    .filter(pl.col("CODIGO META") != "" )
    .rename({"VALOR DEL PAGO":"RP"})# RP no es el pago, dejo RP porque voy a concatenar con las otras ejecuciones de 2025 
                                    # y requiero que tengan el mismo nombre de columna
)

ejecucion_2025_ads_regalias = (
    pl.read_excel(
        archivo_regalias_2025_ads,
        table_name="RegaliasAguasDeSucre"
    )
    .select("CODIGO DE META", "CLASIFICACIÓN RECURSOS", "PAGOS")
    .rename({"CODIGO DE META":"CODIGO META","PAGOS":"RP"}) # RP no es el pago, dejo RP porque voy a concatenar con las otras ejecuciones de 2025 
                                    # y requiero que tengan el mismo nombre de columna
    .with_columns(pl.col("CODIGO META").fill_null(pl.lit("")))
    .filter(pl.col("CODIGO META") != "" )
)

ejecucion_2025_gestiones = (
    pl.read_excel(
        archivo_gestiones_2025,
        table_name="EjecucionGestiones"
    )
    .rename({"EJECUCION FINANCIERA":"RP"}) # RP no es el pago, dejo RP porque voy a concatenar con las otras ejecuciones de 2025 
                                    # y requiero que tengan el mismo nombre de columna
)

ejecucion_pdet_2025 = (
    pl.read_excel(
        archivo_2025_hacienda,
        table_name="EjecucionPDET",
    )
    .select("EJECUCION FINANCIERA", "CODIGO META", "CLASIFICACIÓN RECURSOS")
    .rename({"EJECUCION FINANCIERA":"RP"}) # RP no es el pago, dejo RP porque voy a concatenar con las otras ejecuciones de 2025 
                                    # y requiero que tengan el mismo nombre de columna
)
   
ejecucion_2025_fondo_mixto = (
    pl.read_excel(
        archivo_fondo_mixto_2025,
        table_name="EjecucionFinancieraFondoMixto"
    )
    .select("CLASIFICACIÓN RECURSOS", "EJECUCION FINANCIERA", "CODIGO META")
    .rename({"EJECUCION FINANCIERA":"RP"}) # RP no es el pago, dejo RP porque voy a concatenar con las otras ejecuciones de 2025 
                                    # y requiero que tengan el mismo nombre de columna
)

ejecucion_2025_indersucre_recursos_propios = (
    pl.read_excel(
        archivo_ejecucion_financiera_inder,
        table_name="EjecucionFinancieraINDERTerritorio"
    )
    .rename({"EJECUCION FINANCIERA":"RP"}) # RP no es el pago, dejo RP porque voy a concatenar con las otras ejecuciones de 2025 
                                    # y requiero que tengan el mismo nombre de columna
)

ejecucion_2025_indersucre_regalias = (
    pl.read_excel(
        archivo_ejecucion_financiera_inder,
        table_name="EjecucionFinancieraINDERRegalias"
    )
    .rename({"EJECUCION FINANCIERA":"RP"}) # RP no es el pago, dejo RP porque voy a concatenar con las otras ejecuciones de 2025 
                                    # y requiero que tengan el mismo nombre de columna
)

#______________________________________________________________________________________________________________________________

# Ejecución Financiera 2026

ejecucion_regalias_2026 = (
    pl.read_excel(
        archivo_2026_regalias,
        table_name="Pagos_Regalias_2026" # Este el nombre de la tabla que se espera y el que debe tener
    ).select(pl.all().name.map(lambda x: x.strip().upper().replace("_X0009_",""))
    ).filter( 
        ( pl.col("ULTIMA FECHA PAGO") >= pl.date(2026,1,1) ) & ( pl.col("ULTIMA FECHA PAGO") <= pl.date(2026,12,31) )
    ).select("PAGO EJECUTADO VALOR","CODIGO META","CLASIFICACIÓN RECURSOS",
    ).rename( # Hago la claridad que pago no es RP, sin embargo, le cambio el nombre a la columna para no generar error cuando haga concat con ejecucion_hacienda_2025
        {"PAGO EJECUTADO VALOR":"RP"}
    ).with_columns(
        pl.col("CODIGO META").fill_null(pl.lit(""))
    ).filter(pl.col("CODIGO META") != "")
    
)

ejecucion_hacienda_2026 = (
    pl.read_excel(
        archivo_2026_hacienda,
        table_name="EjecucionHacienda2026", # Este el nombre de la tabla que se espera y el que debe tener
    ).with_columns(
        pl.col("PROYECTO ARCHIVADO","CODIGO META","CLASIFICACIÓN RECURSOS","SE VA A CARGAR EN PI").fill_null(pl.lit("")),
        pl.when(pl.col("DISTRIBUIR DE FORMA EQUITATIVA") == "SI").then(pl.col("RP")/2).otherwise(pl.col("RP"))
    ).filter(
        pl.col("PROYECTO ARCHIVADO") == "",
        pl.col("CODIGO META") != "",
        pl.col("CLASIFICACIÓN RECURSOS") != "",
        pl.col("SE VA A CARGAR EN PI") == "",
    ).select("CODIGO META", "CLASIFICACIÓN RECURSOS", "RP")
)

# Ejecución Financiera 2027. Por lo pronto no es necesario, ya que estamos empezando 2026 y no sé si para ese entonces vaya a cambiar la metodologia

# Esta es la ejecución financiera de 2024 despues de hacer las transformaciones

ejecucion_financ_2024 = (
    pl.concat(
        [ejecucion_regalias_2024,ejecucion_hacienda_2024],how="diagonal"
    ).group_by("CODIGO META").agg(
        pl.col("RP").sum().alias("Ejecución Financiera 2024")
    )
)

# Esta es la ejecución financiera de 2025 despues de hacer las transformaciones

ejecucion_financ_2025 = (
    pl.concat(
        [ejecucion_regalias_2025,ejecucion_hacienda_2025,
         ejecucion_2025_ads_recursos_propios,ejecucion_2025_ads_regalias,
         ejecucion_2025_gestiones,
        ejecucion_pdet_2025,
        ejecucion_2025_fondo_mixto,
        ejecucion_2025_indersucre_recursos_propios,
        ejecucion_2025_indersucre_regalias],how="diagonal"
    )
    .with_columns(pl.col("CODIGO META").str.split(" | "))
).explode("CODIGO META")

ejecucion_financ_2025 = (
        ejecucion_financ_2025.group_by("CODIGO META").agg(
        pl.col("RP").sum().alias("Ejecución Financiera 2025")
    )
)

# Esta es la ejecución financiera de 2026 despues de hacer las transformaciones

ejecucion_financ_2026 = (
    pl.concat(
        [ejecucion_regalias_2026,ejecucion_hacienda_2026],how="diagonal"
    ).group_by("CODIGO META").agg(
        pl.col("RP").sum().alias("Ejecución Financiera 2026")
    )
)

columnas_programacion_financiera = (
    plan_indicativo.select("Codigo Meta",cs.starts_with("Programación").cast(pl.Float64))
    .select(pl.all().name.map(lambda x: x.strip().lower()))
    .select("codigo meta",(programacion_financiera("24")).alias("Programación Financiera 2024"),
            (programacion_financiera("25")).alias("Programación Financiera 2025"),
           (programacion_financiera("26")).alias("Programación Financiera 2026"),(programacion_financiera("27")).alias("Programación Financiera 2027")
    ).join(ejecucion_financ_2024,left_on="codigo meta",right_on="CODIGO META",how="left" # Agrego la ejecución de 2024
    ).join(ejecucion_financ_2025,left_on="codigo meta",right_on="CODIGO META",how="left" # Agrego la ejecución de 2025
    ).join(ejecucion_financ_2026,left_on="codigo meta",right_on="CODIGO META",how="left" # Agrego la ejecución de 2026
    ).with_columns(pl.col("Ejecución Financiera 2024","Ejecución Financiera 2025","Ejecución Financiera 2026").fill_null(pl.lit(0)))
)

Could not determine dtype for column 21, falling back to string
Could not determine dtype for column 22, falling back to string
Could not determine dtype for column 23, falling back to string
Could not determine dtype for column 16, falling back to string
Could not determine dtype for column 21, falling back to string
Could not determine dtype for column 22, falling back to string
Could not determine dtype for column 23, falling back to string


### Ejecución Financiera por Clasificación de Recurso y Tipo de Fuente

In [68]:
orden_fuentes = pl.DataFrame(
    {
        "Clasificación Recursos":["COFINANCIACIÓN MUNICIPIO","ICDE","OTRAS FUENTES","SGP APSB","SGP SALUD","SGP EDUCACION",
            "REGALÍAS","COFINANCIACIÓN NACIÓN","ICLD","CREDITO"],
        "Orden":[1,5,3,7,9,8,10,2,6,4],
        "Tipo Fuente":["Otras Fuentes","Recursos Propios","Otras Fuentes","Sistema General de Participaciones (SGP)",
            "Sistema General de Participaciones (SGP)","Sistema General de Participaciones (SGP)","Sistema General de Regalías",
            "Otras Fuentes","Recursos Propios","Recursos del Crédito"]
    }
)

# Programación de fuentes. 

prog_financ_tipo = (
    plan_indicativo.select(
        cs.starts_with("Programación")
    ).select(
        pl.all().name.map(lambda x: x.strip().lower())
    ).select(
        cs.exclude("programación total24","programación total25","programación total26","programación total27")
    ).unpivot(
        on=cs.numeric(),variable_name="Clasificación Recursos",value_name="Programación financiera"
    ).group_by("Clasificación Recursos").agg(
        pl.col("Programación financiera").sum()
    ).with_columns(
        pl.col("Clasificación Recursos").str.slice(-2).alias("Vigencia"),
        pl.col("Clasificación Recursos").str.replace_all("programación ","").str.replace_all(r"(24|25|26|27)","")
    ).with_columns(
        (pl.lit("Programación Financiera 20") + pl.col("Vigencia")).alias("Vigencia")
    ).pivot(
        index="Clasificación Recursos",on="Vigencia",aggregate_function="sum"
    ).with_columns(
        pl.col(
            "Clasificación Recursos"
        ).str.replace_many(["recursos propios icde","cofinanciación nación","sgp educación","cofinanciación municipio","sgp salud",
                            "sgp apsb","otras fuentes","regalías","recursos propios icld","crédito"],
                           ["ICDE","COFINANCIACIÓN NACIÓN","SGP EDUCACION","COFINANCIACIÓN MUNICIPIO","SGP SALUD","SGP APSB",
                            "OTRAS FUENTES","REGALÍAS","ICLD","CREDITO"]
                          )
    )
)

ejecuciones_financieras = {
    "2024":[ejecucion_regalias_2024,ejecucion_hacienda_2024],
    "2025":[ejecucion_regalias_2025,ejecucion_hacienda_2025,
         ejecucion_2025_ads_recursos_propios,ejecucion_2025_ads_regalias,
         ejecucion_2025_gestiones,
        ejecucion_pdet_2025,
        ejecucion_2025_fondo_mixto,
        ejecucion_2025_indersucre_recursos_propios,
        ejecucion_2025_indersucre_regalias],
    "2026":[ejecucion_regalias_2026,ejecucion_hacienda_2026]
}

terminos = {
    "2024":"24",
    "2025":"25",
    "2026":"26",
    "2027":"27",
}

# ________________________________________________ Esta sección debe poder verse de firma interactiva _____________________________________

filtro_vigencia = "2025" # El filtro de vigencia puede ser un desplegable de selección multiple, 
                        # para que el usario puede seleccionar un periodo o mas o ver todos

try:
    # Concatenar DataFrames de la lista
    ejecucion_financ_tipo = (
        orden_fuentes.join(
        pl.concat(ejecuciones_financieras[filtro_vigencia],how="diagonal"),left_on="Clasificación Recursos",right_on="CLASIFICACIÓN RECURSOS",how="left"
        ).group_by("Clasificación Recursos").agg(
            pl.col("RP").sum().alias(f"Ejecución Financiera {filtro_vigencia}")
        ).join(orden_fuentes,on="Clasificación Recursos",how="inner"
        ).join(prog_financ_tipo,left_on="Clasificación Recursos",right_on="Clasificación Recursos",how="inner"
        ).select("Orden","Tipo Fuente","Clasificación Recursos",f"Programación Financiera {filtro_vigencia}",
                 f"Ejecución Financiera {filtro_vigencia}",
        ).with_columns(
            ( pl.when(pl.col(f"Programación Financiera {filtro_vigencia}")==0
            ).then(pl.lit(0)
            ).otherwise(
                pl.col(f"Ejecución Financiera {filtro_vigencia}") / 
                pl.col(f"Programación Financiera {filtro_vigencia}") )
            ).alias("Porcentaje de Ejecución Financiera")
        ).sort(by="Orden")
    )
    
except ValueError as e:
    print(f"Error de valor: {e}")
    print(f"Verifique que 'ejecuciones_financieras[{filtro_vigencia}]' sea una lista de DataFrames")
    ejecucion_financ_tipo = pl.DataFrame()  # DataFrame vacío como fallback

except KeyError as e:
    print(f"Clave no encontrada: {e}")
    print(f"Verifique que '{filtro_vigencia}' exista en 'ejecuciones_financieras' y 'terminos'")
    ejecucion_financ_tipo = pl.DataFrame()

except Exception as e:
    print(f"Error inesperado: {type(e).__name__}: {e}")
    ejecucion_financ_tipo = pl.DataFrame()

# Acumulado por clasificación de recursos

ejecucion_2024_agrp = (
    pl.concat(ejecuciones_financieras["2024"],how="diagonal"
    ).group_by("CLASIFICACIÓN RECURSOS"
    ).agg(pl.col("RP").sum().alias(f"Ejecución Financiera 2024") # Agrupo la ejecución de 2024
    )
)

ejecucion_2025_agrp = (
    pl.concat(ejecuciones_financieras["2025"],how="diagonal"
    ).with_columns(pl.col("CODIGO META").str.split(" | "))
).explode("CODIGO META")

ejecucion_2025_agrp = (
    ejecucion_2025_agrp.group_by("CLASIFICACIÓN RECURSOS"
    ).agg(pl.col("RP").sum().alias(f"Ejecución Financiera 2025") # Agrupo la ejecución de 2025
    )
)


ejecucion_2026_agrp = (
    pl.concat(ejecuciones_financieras["2026"],how="diagonal"
    ).group_by("CLASIFICACIÓN RECURSOS"
    ).agg(pl.col("RP").sum().alias(f"Ejecución Financiera 2026") # Agrupo la ejecución de 2025
    )
)

ejecucion_financ_acuml_tipo = (
    orden_fuentes.join(
        ejecucion_2024_agrp,left_on="Clasificación Recursos",right_on="CLASIFICACIÓN RECURSOS",how="left"
    ).join(
        ejecucion_2025_agrp,left_on="Clasificación Recursos",right_on="CLASIFICACIÓN RECURSOS",how="left"
    ).join(
        ejecucion_2026_agrp,left_on="Clasificación Recursos",right_on="CLASIFICACIÓN RECURSOS",how="left"
    ).join(
        prog_financ_tipo,on="Clasificación Recursos"
    ).with_columns(
        pl.col(f"Ejecución Financiera 20{terminos["2024"]}",
               f"Ejecución Financiera 20{terminos["2025"]}",
               f"Ejecución Financiera 20{terminos["2026"]}").fill_null(pl.lit(0))
    ).with_columns(
        ( pl.col(f"Ejecución Financiera 20{terminos["2024"]}") + 
          pl.col(f"Ejecución Financiera 20{terminos["2025"]}") + 
          pl.col(f"Ejecución Financiera 20{terminos["2026"]}" )).alias("Ejecución Financiera Acumulada")
    ).with_columns(
        ( pl.col(f"Programación Financiera 20{terminos["2024"]}") + 
          pl.col(f"Programación Financiera 20{terminos["2025"]}") + 
          pl.col(f"Programación Financiera 20{terminos["2026"]}") +
          pl.col(f"Programación Financiera 20{terminos["2027"]}")).alias("Programación Cuatrienio")
    ).select("Orden","Tipo Fuente","Clasificación Recursos","Programación Financiera 2024","Programación Financiera 2025",
             "Programación Financiera 2026","Ejecución Financiera 2024","Ejecución Financiera 2025","Ejecución Financiera 2026",
             "Programación Cuatrienio","Ejecución Financiera Acumulada"
    ).sort(by="Orden")
)

# ____________________________________________________________________________________________________________________________________________

### Ejecución General por Vigencia y General Acumulada

In [69]:
# Ejecución general por vigencia

# ________________________________________________ Esta sección debe poder verse de firma interactiva _____________________________________

ejecucion_general_vigencia = (
    ejecucion_financ_tipo
    .select(
        pl.col(f"Programación Financiera {filtro_vigencia}").sum(),
        pl.col(f"Ejecución Financiera {filtro_vigencia}").sum(),
        (
            pl.col(f"Ejecución Financiera {filtro_vigencia}").sum() / pl.col(f"Programación Financiera {filtro_vigencia}").sum()
        ).alias("Porcentaje de Ejecución Financiera")
    )
)

# Ejecución General Acumulada

ejecucion_general_acumulada = (
    ejecucion_financ_acuml_tipo
    .select(
        pl.col("Programación Cuatrienio").sum(),
        pl.col("Ejecución Financiera Acumulada").sum(),
        (
            pl.col("Ejecución Financiera Acumulada").sum() / pl.col("Programación Cuatrienio").sum()
        ).alias("Porcentaje de Ejecución Acumulada")
    )
)

# ___________________________________________________________________________________________________________________________________________

### Ejecución Financiera por Categoría del Plan de Desarrollo

In [70]:
# Estos calculos se utilizan en caso de que el usuario decida filtrar para ver la programación en esta vigencia

prog_fisica_financiera = (
    columnas_prog_ejec_fisica.join(columnas_programacion_financiera,left_on="Codigo Meta",right_on="codigo meta",how="left")
    .with_columns("Meta Física Esperada 2024","Meta Física Esperada 2025","Meta Física Esperada 2026","Meta Física Esperada 2027").fill_null(pl.lit(0))
)

# ________________________________________________ Esta sección debe poder verse de firma interactiva _____________________________________

prog_financiera_linea_estrategica = (
    prog_fisica_financiera
    .group_by("Línea Estratégica").agg(pl.col(f"Programación Financiera {filtro_vigencia}").sum(),
                                       pl.col(f"Ejecución Financiera {filtro_vigencia}").sum())
    .join(orden_lineas_pdd,on="Línea Estratégica",how="inner")
    .with_columns(         
        ( pl.when(pl.col(f"Programación Financiera {filtro_vigencia}")==0
            ).then(pl.lit(0)
            ).otherwise(
                pl.col(f"Ejecución Financiera {filtro_vigencia}") / 
                pl.col(f"Programación Financiera {filtro_vigencia}") )
            ).alias("Porcentaje de Ejecución Financiera")
    )
    .sort("Orden Linea")
    .select("Orden Linea","Línea Estratégica",f"Programación Financiera {filtro_vigencia}",
            f"Ejecución Financiera {filtro_vigencia}","Porcentaje de Ejecución Financiera")
)

prog_financiera_sector_pdd = (
    prog_fisica_financiera
    .group_by("Sector PDD").agg(pl.col(f"Programación Financiera {filtro_vigencia}").sum(),
                                pl.col(f"Ejecución Financiera {filtro_vigencia}").sum())
    .join(orden_sectores_pdd,on="Sector PDD",how="inner")
    .with_columns(
        ( pl.when(pl.col(f"Programación Financiera {filtro_vigencia}")==0
            ).then(pl.lit(0)
            ).otherwise(
                pl.col(f"Ejecución Financiera {filtro_vigencia}") / 
                pl.col(f"Programación Financiera {filtro_vigencia}") )
            ).alias("Porcentaje de Ejecución Financiera")
    )
    .sort("Orden Sector")
    .select("Orden Sector","Sector PDD",f"Programación Financiera {filtro_vigencia}",
            f"Ejecución Financiera {filtro_vigencia}","Porcentaje de Ejecución Financiera")
)

prog_financiera_programa_pdd = (
    prog_fisica_financiera
    .group_by("Programa PDD").agg(pl.col(f"Programación Financiera {filtro_vigencia}").sum(),
                                  pl.col(f"Ejecución Financiera {filtro_vigencia}").sum())
    .join(orden_programas_pdd,on="Programa PDD",how="inner")
    .with_columns(         
        ( pl.when(pl.col(f"Programación Financiera {filtro_vigencia}")==0
            ).then(pl.lit(0)
            ).otherwise(
                pl.col(f"Ejecución Financiera {filtro_vigencia}") / 
                pl.col(f"Programación Financiera {filtro_vigencia}") )
            ).alias("Porcentaje de Ejecución Financiera")
    )
    .sort("Orden Programa PDD")
    .select("Orden Programa PDD","Programa PDD",f"Programación Financiera {filtro_vigencia}",
            f"Ejecución Financiera {filtro_vigencia}","Porcentaje de Ejecución Financiera")
)

# _________________________________________________________________________________________________________________________________________

# Reporte de Ejecución Física

### Ejecución por Dependencia

In [71]:
ejecucion_por_dependencia_acumulada = (
    prog_fisica_financiera.select(pl.col("Responsable").str.strip_chars(),"PORCENTAJE DE EJECUCIÓN ACUMULADA")
    .group_by("Responsable").agg(
        pl.col("PORCENTAJE DE EJECUCIÓN ACUMULADA").fill_null(pl.lit(0)).mean().alias("Porcentaje de Ejecución Acumulada")
    )
) # Esta variable la construí porque el porcentaje de ejecución acumulado debe calcularse sin el filtro de meta programada de la vigencia

# ________________________________________________ Esta sección debe poder verse de firma interactiva _____________________________________

ejecucion_por_dependencia = (
    prog_fisica_financiera.select(pl.col("Responsable").str.strip_chars(),f"Meta Física Esperada {filtro_vigencia}",
                                  f"PORCENTAJE DE EJECUCIÓN {filtro_vigencia}",f"CATEGORÍA DE EJECUCIÓN FÍSICA {filtro_vigencia}")
    .filter(pl.col(f"Meta Física Esperada {filtro_vigencia}").fill_null(pl.lit(0)) != 0 )
    .with_columns( 
        (pl.when(pl.col(f"Meta Física Esperada {filtro_vigencia}") != 0 )
        .then(pl.lit(1))
        .otherwise(pl.lit(0))).alias(f"Metas Programadas {filtro_vigencia}"),
        (pl.when(pl.col(f"CATEGORÍA DE EJECUCIÓN FÍSICA {filtro_vigencia}") == "Superior" )
        .then(pl.lit(1))
        .otherwise(pl.lit(0))).alias(f"Metas Cumplidas al 100% {filtro_vigencia}"))
    .group_by("Responsable").agg(
        pl.col(f"PORCENTAJE DE EJECUCIÓN {filtro_vigencia}").fill_null(pl.lit(0)).mean().alias(f"Porcentaje de Ejecución {filtro_vigencia}"),
        pl.col(f"Metas Programadas {filtro_vigencia}").sum(),
        pl.col(f"Metas Cumplidas al 100% {filtro_vigencia}").sum()
    )
    .join(homologacion_secretarias,left_on="Responsable",right_on="Responsable en PI",how="left") # La columnas varias secretraías puede quedar como filtro
    .join(ejecucion_por_dependencia_acumulada,on="Responsable",how="left")
    .select("Varias Secretarías","Dependencia Responsable",f"Metas Programadas {filtro_vigencia}",f"Metas Cumplidas al 100% {filtro_vigencia}",
            f"Porcentaje de Ejecución {filtro_vigencia}","Porcentaje de Ejecución Acumulada")
)

# _________________________________________________________________________________________________________________________________________

### Distribución de Metas PDD

In [72]:
programacion_cuatrienio = prog_fisica_financiera.select(pl.col("Meta de cuatrienio").sum()).item()
distribucion_2024 = prog_fisica_financiera.select(pl.col("Meta Física Esperada 2024").sum() / programacion_cuatrienio ).item()
distribucion_2025 = prog_fisica_financiera.select(pl.col("Meta Física Esperada 2025").sum() / programacion_cuatrienio ).item()
distribucion_2026 = prog_fisica_financiera.select(pl.col("Meta Física Esperada 2026").sum() / programacion_cuatrienio ).item()
distribucion_2027 = prog_fisica_financiera.select(pl.col("Meta Física Esperada 2027").sum() / programacion_cuatrienio ).item()

# ________________________________________________ Esta sección debe poder verse de firma interactiva _____________________________________

distribucion_metas_pdd = {
    "2024":distribucion_2024,
    "2025":distribucion_2025,
    "2026":distribucion_2026,
    "2027":distribucion_2027
}
# _________________________________________________________________________________________________________________________________________

### Ejecución Física por Categorías

In [73]:
numero_total_metas = prog_fisica_financiera.get_column("Codigo Meta").count()
numero_metas_programadas_vigencia = (
    prog_fisica_financiera.filter(pl.col(f"Meta Física Esperada {filtro_vigencia}")!=0)
    .get_column("Codigo Meta").count()
)

promedio_programas_con_programacion = (
    prog_fisica_financiera.filter(pl.col(f"Meta Física Esperada {filtro_vigencia}")!=0)
    .group_by("Programa PDD").agg(pl.col(f"PORCENTAJE DE EJECUCIÓN {filtro_vigencia}").mean()).
    rename({f"PORCENTAJE DE EJECUCIÓN {filtro_vigencia}":"Promedio de avance de ejecución de la vigencia"})
)

numero_metas_lineas_con_programacion = (
    prog_fisica_financiera
    .filter(pl.col(f"Meta Física Esperada {filtro_vigencia}")!=0)
    .group_by("Línea Estratégica").agg(pl.col("Codigo Meta").len())
    .rename({"Codigo Meta":"Total Indicadores de Producto con Programacion"})
)

numero_metas_lineas = (
    prog_fisica_financiera
    .group_by("Línea Estratégica").agg(pl.col("Codigo Meta").len())
    .rename({"Codigo Meta":"Total Indicadores de Producto"})
)

numero_metas_sectores_con_programacion = (
    prog_fisica_financiera
    .filter(pl.col(f"Meta Física Esperada {filtro_vigencia}")!=0)
    .group_by("Sector PDD").agg(pl.col("Codigo Meta").len())
    .rename({"Codigo Meta":"Total Indicadores de Producto con Programacion"})
)

numero_metas_sectores = (
    prog_fisica_financiera
    .group_by("Sector PDD").agg(pl.col("Codigo Meta").len())
    .rename({"Codigo Meta":"Total Indicadores de Producto"})
)

numero_metas_programas = (
    prog_fisica_financiera
    .group_by("Programa PDD").agg(pl.col("Codigo Meta").len())
    .rename({"Codigo Meta":"Total Indicadores de Producto"})
)

# ________________________________________________ Esta sección debe poder verse de firma interactiva _____________________________________

ponderado_vigencia = ( # Este ponderado arroja el porcentaje de ejecución de los programas PDD de la vigencia
    prog_fisica_financiera
    .with_columns( 
        (pl.when(pl.col(f"Meta Física Esperada {filtro_vigencia}") != 0 )
        .then(pl.lit(1))
        .otherwise(pl.lit(0))).alias(f"Metas Programadas {filtro_vigencia}") )
    .group_by("Línea Estratégica","Sector PDD","Programa PDD").agg(
        pl.col(f"Metas Programadas {filtro_vigencia}").sum())
    .with_columns( (pl.col(f"Metas Programadas {filtro_vigencia}") / numero_metas_programadas_vigencia).alias("Sobre Numero Total de Metas Programadas") )
    .join(promedio_programas_con_programacion,on="Programa PDD",how="left")
    .with_columns(pl.col("Promedio de avance de ejecución de la vigencia").fill_null(pl.lit(0)))
    .rename({f"Metas Programadas {filtro_vigencia}":"Total Indicadores de Producto Programados"})
)

ponderado_cuatrienio = ( # Este ponderado arroja el porcentaje de ejecución de los programas PDD del cuatrienio
    prog_fisica_financiera
    .group_by("Línea Estratégica","Sector PDD","Programa PDD").agg(
        pl.col("PORCENTAJE DE EJECUCIÓN ACUMULADA").fill_null(pl.lit(0)).mean())
    .join(numero_metas_programas,on="Programa PDD")
    .with_columns( (pl.col("Total Indicadores de Producto") / numero_total_metas).alias("Sobre Numero Total de Metas") )
    .rename({"PORCENTAJE DE EJECUCIÓN ACUMULADA":"Promedio de avance de ejecución acumulada"})
)

avance_vigencia_ponderado = ponderado_vigencia.select(
    pl.col("Promedio de avance de ejecución de la vigencia") * pl.col("Sobre Numero Total de Metas Programadas")
).sum().item()

avance_vigencia_cuatrienio = ponderado_cuatrienio.select(
    pl.col("Promedio de avance de ejecución acumulada") * pl.col("Sobre Numero Total de Metas")
).sum().item()

avance_vigencia_lineas = (
    ponderado_vigencia
    .group_by("Línea Estratégica").agg(
        (pl.col("Promedio de avance de ejecución de la vigencia")
         * pl.col("Sobre Numero Total de Metas Programadas") ).sum() )
    .rename({"Promedio de avance de ejecución de la vigencia":"% Aporte Cumplimiento PDD"})
    .join(numero_metas_lineas_con_programacion,on="Línea Estratégica")
    .with_columns( 
        ( pl.col("Total Indicadores de Producto con Programacion") / numero_metas_programadas_vigencia ).alias("Sobre Numero Total de Indicadores")
    ).with_columns(
        ( pl.col("% Aporte Cumplimiento PDD") / pl.col("Sobre Numero Total de Indicadores") ).alias("% Eficacia Operativa")
    ) # Se puede mostrar todo, pero se grafica principalmente con las columnas: Linea Estrategica y % Eficacia Operativa
)

avance_cuatrienio_lineas = (
    ponderado_cuatrienio
    .group_by("Línea Estratégica").agg(
        (pl.col("Promedio de avance de ejecución acumulada")
         * pl.col("Sobre Numero Total de Metas") ).sum() )
    .rename({"Promedio de avance de ejecución acumulada":"% Aporte Cumplimiento PDD"})
    .join(numero_metas_lineas,on="Línea Estratégica")
    .with_columns( 
        ( pl.col("Total Indicadores de Producto") / numero_total_metas ).alias("Sobre Numero Total de Indicadores")
    ).with_columns(
        ( pl.col("% Aporte Cumplimiento PDD") / pl.col("Sobre Numero Total de Indicadores") ).alias("% Eficacia Operativa")
    ) # Se puede mostrar todo, pero se grafica principalmente con las columnas: Linea Estrategica y % Eficacia Operativa
)

avance_vigencia_sectores = (
    ponderado_vigencia
    .group_by("Sector PDD").agg(
        (pl.col("Promedio de avance de ejecución de la vigencia")
         * pl.col("Sobre Numero Total de Metas Programadas") ).sum() )
    .rename({"Promedio de avance de ejecución de la vigencia":"% Aporte Cumplimiento PDD"})
    .join(numero_metas_sectores_con_programacion,on="Sector PDD")
    .with_columns( 
        ( pl.col("Total Indicadores de Producto con Programacion") / numero_metas_programadas_vigencia ).alias("Sobre Numero Total de Indicadores")
    ).with_columns(
        ( pl.col("% Aporte Cumplimiento PDD") / pl.col("Sobre Numero Total de Indicadores") ).alias("% Eficacia Operativa")
    ) # Se puede mostrar todo, pero se grafica principalmente con las columnas: Sector PDD y % Eficacia Operativa
)

avance_cuatrienio_sectores = (
    ponderado_cuatrienio
    .group_by("Sector PDD").agg(
        (pl.col("Promedio de avance de ejecución acumulada")
         * pl.col("Sobre Numero Total de Metas") ).sum() )
    .rename({"Promedio de avance de ejecución acumulada":"% Aporte Cumplimiento PDD"})
    .join(numero_metas_sectores,on="Sector PDD")
    .with_columns( 
        ( pl.col("Total Indicadores de Producto") / numero_total_metas ).alias("Sobre Numero Total de Indicadores")
    ).with_columns(
        ( pl.col("% Aporte Cumplimiento PDD") / pl.col("Sobre Numero Total de Indicadores") ).alias("% Eficacia Operativa")
    ) # Se puede mostrar todo, pero se grafica principalmente con las columnas: Sector PDD y % Eficacia Operativa
)

avance_cuatrienio_dependencia = (
    prog_fisica_financiera
    .group_by(pl.col("Responsable").str.strip_chars()).agg(pl.col("PORCENTAJE DE EJECUCIÓN ACUMULADA").fill_null(pl.lit(0)).mean())
)

avance_por_dependencia = (
    prog_fisica_financiera
    .filter(pl.col(f"Meta Física Esperada {filtro_vigencia}")!=0)
    .with_columns( 
        (pl.when(pl.col(f"CATEGORÍA DE EJECUCIÓN FÍSICA {filtro_vigencia}") == "Superior" )
        .then(pl.lit(1))
        .otherwise(pl.lit(0))).alias(f"Metas Cumplidas al 100% {filtro_vigencia}") )
    .group_by(pl.col("Responsable").str.strip_chars()).agg(
        pl.col(f"PORCENTAJE DE EJECUCIÓN {filtro_vigencia}").mean(),
        pl.col("Codigo Meta").len(),
        pl.col(f"Metas Cumplidas al 100% {filtro_vigencia}").sum()
    )
    .join(homologacion_secretarias,left_on="Responsable",right_on="Responsable en PI")
    .join(avance_cuatrienio_dependencia,on="Responsable")
    .rename({f"PORCENTAJE DE EJECUCIÓN {filtro_vigencia}":f"Porcentaje de Ejecución {filtro_vigencia}",
             "PORCENTAJE DE EJECUCIÓN ACUMULADA":"Porcentaje de Ejecución Acumulada","Codigo Meta":f"Número de Metas Programadas {filtro_vigencia}"})
    .select("Varias Secretarías","Dependencia Responsable",f"Número de Metas Programadas {filtro_vigencia}",f"Metas Cumplidas al 100% {filtro_vigencia}",
            f"Porcentaje de Ejecución {filtro_vigencia}","Porcentaje de Ejecución Acumulada")
)

# Proyectos

In [74]:
from xlsxwriter import Workbook

def extraer(expr: pl.Expr, patron: str) -> pl.Expr:
    return expr.str.extract(patron, group_index=1).str.strip_chars()


def normalizar_numero(expr: pl.Expr) -> pl.Expr:
    x = expr.str.strip_chars().str.replace_all(r"\s+", "")

    return (
        pl.when(x.is_null() | (x == ""))
        .then(pl.lit(None))
        .when(x.str.contains(r"^\d{1,3}(?:\.\d{3})+,\d+$"))
        .then(
            x.str.replace_all(r"\.", "")
            .str.replace_all(",", ".")
        )
        .when(x.str.contains(r"^\d{1,3}(?:,\d{3})+\.\d+$"))
        .then(x.str.replace_all(",", ""))
        .when(x.str.contains(r"^\d{1,3}(?:\.\d{3})+$"))
        .then(x.str.replace_all(r"\.", ""))
        .when(x.str.contains(r"^\d{1,3}(?:,\d{3})+$"))
        .then(x.str.replace_all(",", ""))
        .when(x.str.contains(r",") & ~x.str.contains(r"\."))
        .then(x.str.replace_all(",", "."))
        .otherwise(x)
        .cast(pl.Float64, strict=False)
    )

patron_bpin = r"\((?i:bpin)\s*:\s*([^()]+?)\s*\)"
patron_tipo_banco = (
    r"\((?i:tipo\s+de\s+banco)\s*:\s*([^()]+?)\s*\)"
)
patron_meta = (
    r"\((?i:(?:"
    r"meta\s+del\s+proyecto|"
    r"meta\s+de\s+la\s+gestion|"
    r"meta\s+de\s+la\s+gestión|"
    r"meta\s+total\s+del\s+indicador|"
    r"meta\s+total\s+de\s+la\s+vigencia|"
    r"meta\s+total\s+del\s+proyecto|"
    r"meta\s+programada|"
    r"meta\s+de\s+la\s+vigencia"
    r"))\s*:\s*([^()]+?)\s*\)"
)
patron_ejecutado = (
    r"\((?i:(?:"
    r"ejecucion\s+2024|"
    r"ejecución\s+2024|"
    r"ejecutado|"
    r"ejecucion|"
    r"ejecución"
    r"))\s*:\s*([^()]+?)\s*\)"
)
patron_estado = (
    r"\((?i:estado\s+en\s+portafolio)\s*:\s*([^()]+?)\s*\)"
)
patron_bloques_info = (
    r"\((?i:(?:"
    r"bpin|"
    r"tipo\s+de\s+banco|"
    r"meta\s+del\s+proyecto|"
    r"meta\s+de\s+la\s+gestión|"
    r"meta\s+de\s+la\s+gestion|"
    r"meta\s+total\s+del\s+indicador|"
    r"meta\s+total\s+de\s+la\s+vigencia|"
    r"meta\s+total\s+del\s+proyecto|"
    r"meta\s+programada|"
    r"meta\s+de\s+la\s+vigencia|"
    r"ejecución\s+2024|"
    r"ejecucion\s+2024|"
    r"ejecutado|"
    r"ejecución|"
    r"ejecucion|"
    r"estado\s+en\s+portafolio"
    r"))\s*:\s*[^()]+?\s*\)"
)

# Cuando escribo solo la palabra proyectos seguida de la vigencia hago referencia a proyectos ejecutado o en ejecuión.
# Como estamos en 2026, uso la columna PROYECTOS/GESTIONES PROGRAMADAS 2026 para saber los proyectos que estan programados, sin embargo
# para conocer los que estan en ejecución utilizo PROYECTOS 2026

filtro_vigencia = ("PROYECTOS 2024","PROYECTOS 2025","PROYECTOS 2026","PROYECTOS/GESTIONES PROGRAMADAS 2026")
proyectos_vigencias = {}

for col_proyecto in filtro_vigencia:
    texto = pl.col(col_proyecto)
    proyectos_ejecutados = (
        prog_fisica_financiera
        .select(
            "Codigo Meta",
            "Línea Estratégica",
            "Sector PDD",
            "Programa PDD",
            "Indicador de producto principal",
            "Código del indicador principal",
            col_proyecto,
        )
        .with_columns(
            pl.col(col_proyecto)
            .fill_null("")
            .cast(pl.String)
            .alias(col_proyecto)
        )
        .filter(
            pl.col(col_proyecto) != "",
            pl.col(col_proyecto) != "0",
        )
        .with_columns(
            pl.col(col_proyecto)
            .str.split("\n\n")
            .alias(col_proyecto)
        )
        .explode(col_proyecto)
        .with_columns(
            pl.col(col_proyecto).str.strip_chars().alias(col_proyecto)
        )
        .filter(
            pl.col(col_proyecto) != "",
            pl.col(col_proyecto) != "0",
        )
        .with_columns(
            texto
            .str.replace_all(patron_bloques_info, "")
            .str.replace_all(r"\s+", " ")
            .str.strip_chars()
            .alias("Nombre del Proyecto"),
    
            extraer(texto, patron_bpin).alias("BPIN"),
    
            extraer(texto, patron_tipo_banco).alias("Tipo de Banco"),
    
            normalizar_numero(extraer(texto, patron_meta)).alias("Meta"),
    
            normalizar_numero(extraer(texto, patron_ejecutado)).alias(
                "Ejecutado"
            ),
    
            extraer(texto, patron_estado).alias("Estado en portafolio"),
        )
        .drop(col_proyecto)
    )
    proyectos_vigencias[col_proyecto] = proyectos_ejecutados

# proyectos_ejecutados.write_excel("PruebaProyectos2025.xlsx",autofit=True)

with Workbook("EjecucionesVigenciasPI_20260517.xlsx") as wb:
    for nombre_hoja, dataframe in proyectos_vigencias.items():
        dataframe.write_excel(workbook=wb, worksheet=f"{nombre_hoja[-14::1]}")